In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
IMG_SIZE=128
BATCH_SIZE=32

train_dir="/Users/lasithcharuka/Desktop/spoof/anti_spoof_prepared/train"
val_dir="/Users/lasithcharuka/Desktop/spoof/anti_spoof_prepared/val"


train_gen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)
val_gen=ImageDataGenerator(rescale=1./255)


In [13]:
import os

print("Training folders:", os.listdir(train_dir))
print("Validation folders:", os.listdir(val_dir))

for folder in ["real", "spoof"]:
    print("Train", folder, "=", len(os.listdir(os.path.join(train_dir, folder))))
    print("Val", folder, "=", len(os.listdir(os.path.join(val_dir, folder))))

Training folders: ['real', 'spoof']
Validation folders: ['real', 'spoof']
Train real = 1223
Val real = 314
Train spoof = 1223
Val spoof = 314


In [14]:
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)
   



Found 2446 images belonging to 2 classes.
Found 628 images belonging to 2 classes.


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(1.0), 1: np.float64(1.0)}


In [16]:
print(train_data.class_indices)

{'real': 0, 'spoof': 1}


In [17]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

/Users/lasithcharuka/Library/Python/3.9/lib/python/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [19]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    class_weight=class_weights
)

/Users/lasithcharuka/Library/Python/3.9/lib/python/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 59s 752ms/step - accuracy: 0.6251 - loss: 2.2380 - val_accuracy: 0.5000 - val_loss: 4.0495
Epoch 2/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 60s 783ms/step - accuracy: 0.7108 - loss: 0.5578 - val_accuracy: 0.6242 - val_loss: 1.7001
Epoch 3/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 58s 750ms/step - accuracy: 0.7709 - loss: 0.4997 - val_accuracy: 0.6417 - val_loss: 1.8560
Epoch 4/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 56s 727ms/step - accuracy: 0.7927 - loss: 0.4698 - val_accuracy: 0.6688 - val_loss: 1.0619
Epoch 5/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 58s 751ms/step - accuracy: 0.8203 - loss: 0.4094 - val_accuracy: 0.5892 - val_loss: 1.2092
Epoch 6/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 57s 738ms/step - accuracy: 0.8004 - loss: 0.4512 - val_accuracy: 0.6433 - val_loss: 0.8991
Epoch 7/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 68s 883ms/step - accuracy: 0.8389 - loss: 0.3653 - val_accuracy: 0.7022 - val_loss: 0.8591
Epoch 8/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 59s 766ms/step - accuracy: 0.8577 - loss: 0.3786 - val_accu